In [1]:
#Zach Brand - Final Project: Dataset Movie Review Phrase Data

In [2]:
#Task 1
#For your choice of dataset, you will first process the text, tokenize it, and choose whether to do further
#pre-processing or filtering. If you do some pre-processing or filtering, then using the text with and without it
#can be one of your experiments. 

#Movie Reviews
import pandas as pd
import nltk
import spacy
import numpy as np
from nltk.corpus import stopwords
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_validate
import random
import collections

nltk.download('stopwords')
nltk.download('vader_lexicon')
nltk.download('averaged_perceptron_tagger_eng')
from nltk.corpus import stopwords
stopwords = set(stopwords.words('english'))

nlp = spacy.load('en_core_web_sm', disable=['parser', 'ner'])  #lemmatizer 

train = pd.read_csv(r'C:\Users\PC\Documents\Classes\IST-664 NLP\Final Project\FinalProjectData\kagglemoviereviews\corpus\train.tsv', delimiter='\t').head(5000)
test = pd.read_csv(r'C:\Users\PC\Documents\Classes\IST-664 NLP\Final Project\FinalProjectData\kagglemoviereviews\corpus\test.tsv', delimiter='\t').head(5000)
sampleS = pd.read_csv(r'C:\Users\PC\Documents\Classes\IST-664 NLP\Final Project\FinalProjectData\kagglemoviereviews\corpus\sampleSubmission.csv')

print(train.head())
print(test.head())
print(sampleS.head())


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\PC\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\PC\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\PC\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


   PhraseId  SentenceId                                             Phrase  \
0         1           1  A series of escapades demonstrating the adage ...   
1         2           1  A series of escapades demonstrating the adage ...   
2         3           1                                           A series   
3         4           1                                                  A   
4         5           1                                             series   

   Sentiment  
0          1  
1          2  
2          2  
3          2  
4          2  
   PhraseId  SentenceId                                             Phrase
0    156061        8545  An intermittently pleasing but mostly routine ...
1    156062        8545  An intermittently pleasing but mostly routine ...
2    156063        8545                                                 An
3    156064        8545  intermittently pleasing but mostly routine effort
4    156065        8545         intermittently pleasing but mostly

In [3]:
#Preprocessing: Used Grok and https://www.geeksforgeeks.org/python-functions/to simplify my code. Created function for tokenization, stopword removal, 
#and punctuation removal in one step. Rather than the five block individual step process I had prior.
def preprocess_phrase(text):
    text = str(text)
    tokens = nltk.tokenize.word_tokenize(text)
    tokens = [t.lower() for t in tokens if t.isalnum()]
    tokens = [t for t in tokens if t not in stopwords]
    doc = nlp(' '.join(tokens))
    lemmas = [token.lemma_ for token in doc if token.is_alpha and token.lemma_.lower() not in stopwords]
    return lemmas

train['lemtokens'] = train['Phrase'].apply(preprocess_phrase)
test['lemtokens'] = test['Phrase'].apply(preprocess_phrase)
print("Train with lemmatized tokens (first 5 rows):\n", train[['Phrase', 'lemtokens', 'Sentiment']].head())
print("Test with lemmatized tokens (first 5 rows):\n", test[['Phrase', 'lemtokens']].head())

Train with lemmatized tokens (first 5 rows):
                                               Phrase  \
0  A series of escapades demonstrating the adage ...   
1  A series of escapades demonstrating the adage ...   
2                                           A series   
3                                                  A   
4                                             series   

                                           lemtokens  Sentiment  
0  [series, escapade, demonstrate, adage, good, g...          1  
1  [series, escapade, demonstrate, adage, good, g...          2  
2                                           [series]          2  
3                                                 []          2  
4                                           [series]          2  
Test with lemmatized tokens (first 5 rows):
                                               Phrase  \
0  An intermittently pleasing but mostly routine ...   
1  An intermittently pleasing but mostly routine ...   
2       

In [4]:
#Task 2
#The second step is to produce the features in the notation of the NLTK. For this you should write
#feature functions in Python. You should start with the "bag-of-words" features where you collect all the words in the corpus
#and select some number of most frequent words to be word features. 
#Now use the NLTK Naive Bayes classifier on your feature sets. You should use cross-validation to obtain precision, recall, and F-measure scores.
#Or you can choose to produce the features as a csv file and use Weka or Sci-Kit learn to train and test a classifier, using cross-validation scores.
from nltk import FreqDist
from nltk.classify import NaiveBayesClassifier
from nltk.metrics.scores import precision, recall, f_measure
import random
import collections

all_lemmas = []
for lemmas in train['lemtokens']:
    for lemma in lemmas:
        all_lemmas.append(lemma)
freqdist = collections.Counter(all_lemmas)
word_features = [word for word, _ in freqdist.most_common(150)]
print("Top word features (first 5):", word_features[:5])


Top word features (first 5): ['movie', 'film', 'make', 'good', 'like']


In [5]:
#Feature Sets
def get_features(lemmas):
    features = {}
    for word in word_features:
        features[word] = (word in lemmas)
    return features

train_features = []
for i in range(len(train)):
    train_features.append((get_features(train['lemtokens'][i]), train['Sentiment'][i]))
test_features = [get_features(test['lemtokens'][i]) for i in range(len(test))]

In [6]:
# Cross-validation (5 folds) https://www.datacamp.com/tutorial/k-fold-cross-validation. Code - Grok/ChatGPT
random.seed(42)
random.shuffle(train_features)
folds = 5
fold_size = len(train_features) // folds
prec_scores = []
rec_scores = []
f_scores = []

for i in range(folds):
    val_set = train_features[i * fold_size:(i + 1) * fold_size]
    train_set = train_features[:i * fold_size] + train_features[(i + 1) * fold_size:]
    classifier = NaiveBayesClassifier.train(train_set)
    val_labels = [label for _, label in val_set]
    val_preds = [classifier.classify(f) for f, _ in val_set]
    for label in set(val_labels):
        ref_set = {j for j, l in enumerate(val_labels) if l == label}  
        test_set = {j for j, p in enumerate(val_preds) if p == label}  
        prec_scores.append(precision(ref_set, test_set) or 0)
        rec_scores.append(recall(ref_set, test_set) or 0)
        f_scores.append(f_measure(ref_set, test_set) or 0)

print("Average Precision:", sum(prec_scores) / len(prec_scores))
print("Average Recall:", sum(rec_scores) / len(rec_scores))
print("Average F-measure:", sum(f_scores) / len(f_scores))

Average Precision: 0.4701387279793648
Average Recall: 0.34440884622412704
Average F-measure: 0.36857840035813866


In [7]:
#Task 3
#For a base level completion of experiments, carry out at least several experiments where you use two different features
#and compare the results. For example, you may take the unigram word features as a baseline and see if the features you
#designed improve the accuracy of the classification. Here are some of the types of experiments that we have done so far:
#A. 
#1. Filter by stop words or other pre-processing methods.
#3. Representing negation (if using twitter data, note the difference in tokenization).
#4. Using a sentiment lexicon with scores or counts: Subjectivity.
#5. Different sizes of vocabularies.
#6. POS tag features

#You must define at least one "new" feature function not given in class. Also, you should try to combine some of the earlier features, e.g.
#to use unigrams, bigrams, POS tag counts, and sentiment word counts all in one feature set. Examples of new features:

#1. Use the LIWC sentiment lexicon.
#2. Combine the use of sentiment lexicons.
#3. Use a different representation of negation, for example, carrying the scope of the negation work over to the next punctuation.
#B. Choose an additional, more advanced type of task from this list, or propose your own.
#1. Using Weka or Sci Kit Learn classifiers with features produced in NLTK. Note that you should not use the built-in vectorizers from Weka or Sci Kit Learn.
#1. Using an additional type of lexicon besides Subjectivity or LIWC.
#3. In addition to using cross-validation on the training set, train the classifier on the entire training set and test it on a separately available test set (only the SemEval data has these).
#Note that you must save the vocabulary from the training set and use the same for creating feature sets for the test data.
#Implement additional features



In [8]:
#Unigrams & Bigrams
all_lemmas = [lemma for lemmas in train['lemtokens'] for lemma in lemmas]
all_bigrams = ['_'.join(bigram) for tokens in train['lemtokens'] for bigram in nltk.bigrams(tokens)]
unigram_freq = collections.Counter(all_lemmas)
bigram_freq = collections.Counter(all_bigrams)
unigram_features_150 = [word for word, _ in unigram_freq.most_common(150)]  # Baseline
unigram_features_1000 = [word for word, _ in unigram_freq.most_common(1000)]  # Experiment 2
bigram_features = [bigram for bigram, _ in bigram_freq.most_common(500)]  # Experiment 3

In [9]:
#Parts of Speech Tagging & Sentiment Lexicon
all_pos = [tag for tokens in train['lemtokens'] for _, tag in nltk.pos_tag(tokens)]
pos_features = list(set(all_pos))  # Unique POS tags (e.g., JJ, NN)
vader = SentimentIntensityAnalyzer()
vader_pos = {word for word, score in vader.lexicon.items() if score > 0}
vader_neg = {word for word, score in vader.lexicon.items() if score < 0}

In [35]:
# Feature functions
def get_unigram_features_150(lemmas):
    return {f'uni_{word}': (word in lemmas) for word in unigram_features_150}

def get_unigram_features_1000(lemmas):
    return {f'uni_{word}': (word in lemmas) for word in unigram_features_1000}

def get_combined_features(lemmas):
    tokens = lemmas
    bigrams = ['_'.join(bigram) for bigram in nltk.bigrams(tokens)]
    pos_tags = [tag for _, tag in nltk.pos_tag(tokens)]
    features = {f'uni_{word}': (word in lemmas) for word in unigram_features_1000}
    features.update({f'bi_{bigram}': (bigram in bigrams) for bigram in bigram_features})
    features.update({f'pos_{tag}': pos_tags.count(tag) for tag in pos_features})
    features['vader_pos'] = sum(1 for word in lemmas if word in vader_pos)
    features['vader_neg'] = sum(1 for word in lemmas if word in vader_neg)
    return features
for i in range(3):
    lemmas = train['lemtokens'].iloc[i]
    print(f"\nPhrase {i+1}: {train['Phrase'].iloc[i]}")
    features = get_combined_features(lemmas)
    for key, value in list(features.items())[:10]:
        print(f"{key}: {value}")


Phrase 1: A series of escapades demonstrating the adage that what is good for the goose is also good for the gander , some of which occasionally amuses but none of which amounts to much of a story .
uni_movie: False
uni_film: False
uni_make: False
uni_good: True
uni_like: False
uni_one: False
uni_story: True
uni_character: False
uni_much: True
uni_well: False

Phrase 2: A series of escapades demonstrating the adage that what is good for the goose
uni_movie: False
uni_film: False
uni_make: False
uni_good: True
uni_like: False
uni_one: False
uni_story: False
uni_character: False
uni_much: False
uni_well: False

Phrase 3: A series
uni_movie: False
uni_film: False
uni_make: False
uni_good: False
uni_like: False
uni_one: False
uni_story: False
uni_character: False
uni_much: False
uni_well: False


In [11]:
#Experiment 1: 1000 Unigrams (Naive Bayes)
train_features_1000 = [(get_unigram_features_1000(row['lemtokens']), row['Sentiment']) for _, row in train.iterrows()]
random.seed(42)
random.shuffle(train_features_1000)
prec_scores = []
rec_scores = []
f_scores = []

for i in range(folds):
    val_set = train_features_1000[i * fold_size:(i + 1) * fold_size]
    train_set = train_features_1000[:i * fold_size] + train_features_1000[(i + 1) * fold_size:]
    classifier = NaiveBayesClassifier.train(train_set)
    val_labels = [label for _, label in val_set]
    val_preds = [classifier.classify(f) for f, _ in val_set]
    for label in set(val_labels):
        ref_set = {j for j, l in enumerate(val_labels) if l == label}
        test_set = {j for j, p in enumerate(val_preds) if p == label}
        prec_scores.append(precision(ref_set, test_set) or 0)
        rec_scores.append(recall(ref_set, test_set) or 0)
        f_scores.append(f_measure(ref_set, test_set) or 0)

print("Average Precision:", sum(prec_scores) / len(prec_scores))
print("Average Recall:", sum(rec_scores) / len(rec_scores))
print("Average F-measure:", sum(f_scores) / len(f_scores))


Average Precision: 0.5223675523105453
Average Recall: 0.4643230511783641
Average F-measure: 0.48457924528017676


In [12]:
#Experiment 2
train_features_combined = [(get_combined_features(row['lemtokens']), row['Sentiment']) for _, row in train.iterrows()]
random.seed(42)
random.shuffle(train_features_combined)
prec_scores = []
rec_scores = []
f_scores = []

for i in range(folds):
    val_set = train_features_combined[i * fold_size:(i + 1) * fold_size]
    train_set = train_features_combined[:i * fold_size] + train_features_combined[(i + 1) * fold_size:]
    classifier = NaiveBayesClassifier.train(train_set)
    val_labels = [label for _, label in val_set]
    val_preds = [classifier.classify(f) for f, _ in val_set]
    for label in set(val_labels):
        ref_set = {j for j, l in enumerate(val_labels) if l == label}
        test_set = {j for j, p in enumerate(val_preds) if p == label}
        prec_scores.append(precision(ref_set, test_set) or 0)
        rec_scores.append(recall(ref_set, test_set) or 0)
        f_scores.append(f_measure(ref_set, test_set) or 0)

print("Average Precision:", sum(prec_scores) / len(prec_scores))
print("Average Recall:", sum(rec_scores) / len(rec_scores))
print("Average F-measure:", sum(f_scores) / len(f_scores))

Average Precision: 0.48802534050235785
Average Recall: 0.45367413024183245
Average F-measure: 0.4634794092786315


In [13]:
#Experiment 3 Advanced Method
def features_to_array(features, all_features):
    return [1 if f in features else 0 for f in all_features]

all_combined_features = sorted(set(f for features, _ in train_features_combined for f in features))
X_train = [features_to_array(features, all_combined_features) for features, _ in train_features_combined]
y_train = [label for _, label in train_features_combined]
test_features_combined = [get_combined_features(row['lemtokens']) for _, row in test.iterrows()]
X_test = [features_to_array(features, all_combined_features) for features in test_features_combined]

clf = LogisticRegression(max_iter=1000, multi_class='multinomial')
scores = cross_validate(clf, X_train, y_train, cv=5, scoring=['precision_macro', 'recall_macro', 'f1_macro'])
print("\nAdvanced Task: Combined Features (Logistic Regression)")
print("Average Precision:", scores['test_precision_macro'].mean())
print("Average Recall:", scores['test_recall_macro'].mean())
print("Average F-measure:", scores['test_f1_macro'].mean())

C:\Users\PC\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
C:\Users\PC\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\PC\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
C:\Users\PC\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precisio


Advanced Task: Combined Features (Logistic Regression)
Average Precision: 0.11676
Average Recall: 0.2
Average F-measure: 0.14744284283134568


C:\Users\PC\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
